# LaGoVAD-STP — first run: **train on Nexar, eval on DoTA-CAP-dev**, 1 seed

Proposal: `core/docs/final/Thesis proposal … .md` §3.2 (STP = SA² in the CLIP branch + learnable text prefix).
Build notes, deviations and the **read-out rule** (written before any number): `core/docs/final/STP_BUILD.md`.

**Host = v5 R1 / v2 Nexar A2:** the motion stream enters as `x_t = c̃_t + W_u (c·ũ_t/σ_u)`, `W_u` zero-init
(the current code), **not** the proposal's `tanh(α)·W·LN(v)` (STP_BUILD D1).

| arm | SA² (CLIP patches) | text prompt | trained here |
|---|---|---|---|
| **R1** (control) | — | released 32 soft prompts | no: v5 Stage 1 run, read in v5 Stage 2 (0.8102) |
| **R4** | ✓ top-12 motion patches | released (prefix **OFF**) | yes |
| **R6** | ✓ | learnable prefix l = 8 (prefix **ON**) | yes |
| R5 (optional) | — | prefix ON | set `ARMS` |

R6 − R4 = the prefix on/off contrast; R4 − R1 = SA² alone. One seed ⇒ every sentence is
**PRELIMINARY (n = 1 seed)** and adopts nothing.

| step | what | GPU? | resumable |
|---|---|---|---|
| 1–3 | setup, deps, helpers | — | |
| 4 | **Nexar patches**: decode training sources, 1×1 + 7×7 window crops, CLS gate, top-12 by Mo (~11 M CLIP crops) | **yes** | per video |
| 5 | **DoTA-CAP-dev patches**: frame archive → windows (`clip_windows archive`) → `select` (needs `dota_cap_pack_frames.ipynb` first) | **yes** | per clip |
| 6 | **train** `ARMS` × s2024 (1,749 steps, A2 recipe + `stp.*`) + config provenance vs v2 A2 | **yes** | J10-verified |
| 7 | protocol B on DoTA-CAP-dev per arm | yes | skip if read |
| 8 | **read-out** → paste `REPORTS/stp_readout.md` to Claude | no | |

**Drive:** code `Thesis-Final/kat-vad` (upload branch `v_final`); everything this notebook writes goes to
`Thesis-Final/` (caches + outputs). `Thesis/`, `Thesis-V2/`, `Thesis-V5/` are read-only.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data (Nexar raw, DoTA, CAP tars), cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, KNN, census, v2 A2 runs, alignment
DRIVE_V5 = '/content/drive/MyDrive/Thesis-V5'     # read-only: R1's DoTA-CAP-dev read (v5 Stage 2)
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every cache / output this notebook writes
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'     # new caches (clip_patches/, clip_windows/) land here
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'STP_ARMS'), 'this upload predates LaGoVAD-STP -- re-upload kat-vad/'
assert (REPO / 'core/docs/final/STP_BUILD.md').is_file(), 'no STP_BUILD.md (the read-out rule) in this upload'

ARMS     = ['R4', 'R6']          # add 'R5' for prefix-only (no patches needed)
SEEDS    = [constants.STP_SEEDS[0]]   # s2024 -> PRELIMINARY (n = 1 seed). list(constants.STP_SEEDS) = full read
CONTROL  = constants.STP_CONTROL_ARM  # R1
ENC      = constants.V5_ENCODER       # V2-S
C        = constants.NEXAR_CONSTRUCTION_WINDOW
DATASET  = constants.NEXAR_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
STRIDE_N, STRIDE_D, K, MOTION = (constants.STP_NEXAR_STRIDE, constants.STP_DOTA_STRIDE,
                                 constants.STP_TOP_K, constants.STP_MOTION_RAW)
assert all(a in constants.STP_ARMS for a in ARMS), ARMS
NEEDS_PATCHES = any(constants.STP_ARMS[a]['sa2'] for a in ARMS)

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
RAW      = constants.DATA_ROOT / DATASET / 'raw'
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
CLIP_S1  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET           # CLS gate for the 1x1 cell
A2_BAKE  = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                         # host rows [x ; c u~/sigma_u]
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
A2_RUNS  = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'A2'                  # config provenance
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
DOTA_A2  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
ARCHIVE  = Path(DRIVE_F) / 'data' / constants.DOTA_CAP_FRAMES_DIRNAME     # dota_cap_pack_frames.ipynb
R1_PB    = Path(DRIVE_V5) / 'outputs' / 'v5' / 'v5_stage2' / 'eval' / 'dota' / CAP_DEV / CONTROL
A2_PB    = V2_OUT / constants.NEXAR_FULL_RUNS_DIR / 'eval' / 'dota' / 'A2' / CAP_DEV / 'protocol_b_readout.json'

from core.data import stp_patches as stp_data      # noqa: E402
from core.tools import clip_windows               # noqa: E402

PATCH_N  = stp_data.cache_dir(DATASET, STRIDE_N, MOTION, K)                 # Thesis-Final/cache/clip_patches/...
WIN_D    = clip_windows.cache_dir(constants.DOTA_CAP_DATASET, STRIDE_D, constants.STP_WINDOW_GRIDS,
                                  constants.STP_CELLS_RESIZED)
PATCH_D  = stp_data.cache_dir(constants.DOTA_CAP_DATASET, STRIDE_D, MOTION, K)
ROOT     = constants.OUTPUT_ROOT / constants.STP_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/stp')
for p in (REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']     # 1,749 = 33 x 53
for name, path in (('Nexar raw', RAW), ('census', CENSUS), ('Nexar_window', DATA), ('clip s1', CLIP_S1),
                   ('A2 bake', A2_BAKE), ('KNN', KNN), ('v2 A2 runs', A2_RUNS), ('DoTA s1', DOTA_S1),
                   ('DoTA-CAP-dev A2 bake', DOTA_A2), ('DoTA labels', DOTA / 'labels_s8'),
                   ('R1 read', R1_PB / 'clip_aucs.json')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:22s} {path}')
print(f'ARMS {ARMS} | SEEDS {SEEDS} | steps {STEPS} ({EPOCHS} epochs)')
print('Nexar patches ->', PATCH_N, '\nDoTA windows  ->', WIN_D, '\nDoTA patches  ->', PATCH_D, '\nruns/reads    ->', ROOT)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn xlrd
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 4-7 need one'
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF (absl eats INFO logs)


def run(cmd, log=None, quiet_steps=False):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'epoch' in line and 'batch' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (or only ``ids`` + manifests) / file to VM disk once (bit-identical; C2)."""
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        tmp = dst.with_name(dst.name + '.partial')
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.copytree(src, tmp)
        os.replace(tmp, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + '.partial')
        shutil.copy2(src, tmp)
        os.replace(tmp, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10, lesson (bi))."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


from core.data.v2_splits import load_split    # noqa: E402
from core.tools.stp_patches import training_sources, video_done   # noqa: E402

CAP_IDS = sorted(load_split(CAP_DEV))
assert len(CAP_IDS) == 569, len(CAP_IDS)
SOURCES = sorted(training_sources(DATA))
print(f'{len(SOURCES)} Nexar training sources | {len(CAP_IDS)} DoTA-CAP-dev clips')

## 4. Nexar patch cache (GPU, resumable per video)

`stp_patches nexar`: one PyAV decode per mp4, only frames `s1[::8]` resized (identical values to `Nexar_s1_ncc`),
1×1 + 7×7 cells of the 224² squash → frozen CLIP under fp16 autocast. **Gate:** the 1×1 cell must reproduce the
video's own `Nexar_s1_ncc[::8]` rows (row count, every row cos ≥ 0.95, mean ≥ 0.99); a failing video stores nothing.
Only the top-12 by Mo is stored (≈ 2.6 GB; the full grid would be ≈ 11 GB). Writes straight to Drive, atomically —
**re-run this cell after a disconnect**. Then every file is checked against the A2 bake's row count (C13).

In [ ]:
assert torch.cuda.is_available(), 'torch sees no GPU'
if NEEDS_PATCHES:
    todo = [v for v in SOURCES if not video_done(PATCH_N, v)]
    print(f'Nexar patches: {len(SOURCES) - len(todo)}/{len(SOURCES)} done')
    if todo:
        t0 = time.time()
        run([sys.executable, '-m', 'core.tools.stp_patches', 'nexar', '--root', RAW, '--census', CENSUS,
             '--data-dir', DATA, '--clip-s1-dir', CLIP_S1, '--stride', STRIDE_N, '--motion', MOTION,
             '--top-k', K, '--amp', '--out-dir', PATCH_N, '--report', REPORTS / 'stp_extract_nexar.json'],
            log=LOCAL / 'extract_nexar.log')
        print(f'extracted in {(time.time() - t0) / 60:.0f} min')
        flush_drive()
    rep = json.loads((REPORTS / 'stp_extract_nexar.json').read_text())['summary'] \
        if (REPORTS / 'stp_extract_nexar.json').exists() else None
    print('extract summary:', rep)
    bad = []
    for v in SOURCES:                      # C13: patch rows == host rows, per source video
        n_p = np.load(PATCH_N / f'{v}.npy', mmap_mode='r').shape
        n_a = np.load(A2_BAKE / f'{v}.npy', mmap_mode='r').shape
        if n_p[0] != n_a[0] or n_p[1] != K * constants.STP_PATCH_DIM:
            bad.append((v, n_p, n_a))
    assert not bad, f'{len(bad)} patch files do not match the A2 bake rows, e.g. {bad[:3]}'
    print(f'Nexar patch cache OK: {len(SOURCES)} videos, rows == A2 bake rows')
else:
    print('no SA² arm in ARMS -- Nexar patches not needed')

## 5. DoTA-CAP-dev patch cache (GPU, resumable per clip), read from the frame archive, no tar stream

Prerequisite: `colab/final/dota_cap_pack_frames.ipynb` (CPU) has run and its verify printed `pass: True`.
`clip_windows archive` unzips each clip to VM disk, encodes 1×1 + 7×7 cells of the 224² squash at stride 3
(the rows of `DoTA_s1_ncc[::3]`), and gates them: the 1×1 cell must reproduce those rows, corpus mean ≥ 0.99.
Then `stp_patches select` keeps the top-12. The window cache (≈ 1 GB) is kept for maps and ablations.
**Re-run after a disconnect.**

In [ ]:
if NEEDS_PATCHES:
    assert (ARCHIVE / constants.DOTA_CAP_FRAMES_MANIFEST).is_file(), \
        f'no frame archive at {ARCHIVE}: run colab/final/dota_cap_pack_frames.ipynb first'
    ids_file = write_ids(LOCAL / 'dota_cap_dev.txt', CAP_IDS)
    todo = [v for v in CAP_IDS if not (WIN_D / f'{v}.npy').exists()]
    print(f'DoTA-CAP-dev windows: {len(CAP_IDS) - len(todo)}/{len(CAP_IDS)} done')
    if todo:
        t0 = time.time()
        run([sys.executable, '-m', 'core.tools.clip_windows', 'archive', '--archive-dir', ARCHIVE,
             '--work-dir', LOCAL / 'unzip', '--dota-clip-dir', DOTA_S1, '--dota-ids-file', ids_file,
             '--report', REPORTS / 'stp_windows_dota.json', '--output-dir', WIN_D,
             '--grids', 1, constants.STP_GRID, '--cells', constants.STP_CELLS_RESIZED,
             '--stride', STRIDE_D, '--device', 'auto'], log=LOCAL / 'windows_dota.log')
        print(f'windows: {(time.time() - t0) / 60:.0f} min')
    report = json.loads((REPORTS / 'stp_windows_dota.json').read_text())
    missing = {v: report.get(v, {}).get('reason', 'not_run') for v in CAP_IDS if not (WIN_D / f'{v}.npy').exists()}
    assert not missing, f'{len(missing)} DoTA-CAP-dev clips have no window cache: {dict(list(missing.items())[:10])}'
    corpus = clip_windows.corpus_gate([report[v] for v in CAP_IDS if v in report])
    print('1x1 corpus gate:', corpus)
    assert corpus['reason'] == 'ok', f'window transform disagrees with DoTA_s1_ncc: {corpus}'
    run([sys.executable, '-m', 'core.tools.stp_patches', 'select', '--windows-dir', WIN_D, '--ids-file', ids_file,
         '--cells', constants.STP_CELLS_RESIZED, '--motion', MOTION, '--top-k', K, '--out-dir', PATCH_D,
         '--report', REPORTS / 'stp_select_dota.json'])
    for v in CAP_IDS:                     # C13: patch rows == the A2 bake's protocol-B rows
        assert np.load(PATCH_D / f'{v}.npy', mmap_mode='r').shape[0] == np.load(DOTA_A2 / f'{v}.npy', mmap_mode='r').shape[0], v
    print(f'DoTA-CAP-dev patch cache OK: {len(CAP_IDS)} clips')
    flush_drive()

## 6. Train `ARMS` × `SEEDS` (GPU), then config provenance

Exactly v5 Stage 1's R1 command (v2 Nexar A2's recipe) **without** the V7 grid, plus the arm's `stp.*` keys and
`--patch-dir` (SA² arms). Local training, atomic publish to Drive, J10 re-verified after a flush, then
`verified.json`; a verified run is never retrained. Training inputs are staged only if a run actually trains.

In [ ]:
INPUTS = {'data': (DATA, LOCAL / 'stage' / f'data_{C}'), 'a2': (A2_BAKE, LOCAL / 'stage' / 'A2_s8'),
          'knn': (KNN, LOCAL / 'stage' / f'knn_{C}.npz'), 'patch_n': (PATCH_N, LOCAL / 'stage' / 'patch_nexar')}
_STAGED = {}


def staged(key):
    if key not in _STAGED:
        _STAGED[key] = stage(*INPUTS[key])
    return _STAGED[key]


def stp_sets(arm):
    return [x for k, v in constants.STP_ARMS[arm].items() for x in ('--set', f'stp.{k}={str(v).lower()}')]


def train_cmd(arm, seed, out_dir):
    cmd = [sys.executable, '-m', 'core.train',
           '--set', 'train.stage=2', '--set', f'train.seed={seed}',
           '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
           '--set', f'data.dataset={DATASET}',
           '--set', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
           '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
           '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
           '--set', 'data.is_egocentric=false',
           '--set', f'v2.motion={ENC}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}', *stp_sets(arm),
           '--data-dir', staged('data'), '--clip-dir', staged('a2'), '--knn-cache', staged('knn'),
           '--output-dir', out_dir]
    if constants.STP_ARMS[arm]['sa2']:
        cmd += ['--patch-dir', staged('patch_n')]
    return cmd


def is_verified(drive_dir):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == STEPS and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def train_arm(arm, seed):
    drive_dir, local = ROOT / arm / f's{seed}', LOCAL / 'runs' / arm / f's{seed}'
    if is_verified(drive_dir):
        print(f'{arm} s{seed}: verified on Drive -- not retrained')
        return
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run(train_cmd(arm, seed, local) + resume, log=LOCAL / f'train_{arm}_s{seed}.log', quiet_steps=True)
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{arm} s{seed}: J10 locally'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, drive_dir / name)
    flush_drive()
    assert ckpt_step(drive_dir / 'checkpoint_last.pt') == STEPS, f'{arm} s{seed}: Drive copy is not step {STEPS}'
    (LOCAL / 'verified.json').write_text(json.dumps(
        {'step': STEPS, 'ckpt_bytes': (drive_dir / 'checkpoint_last.pt').stat().st_size}))
    publish(LOCAL / 'verified.json', drive_dir / 'verified.json')
    print(f'{arm} s{seed}: {(time.time() - t0) / 60:.0f} min | last mil {last.get("mil"):.4f} | '
          f'SA² entropy {last.get("stp_entropy")} | |W_q| {last.get("stp_wq_norm")} | prompt {last.get("prompt_norm")}')


assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
for seed in SEEDS:
    for arm in ARMS:
        train_arm(arm, seed)

# provenance (STP_BUILD D7): config == v2 Nexar A2's of the same seed except `stp` (and `v5`, which
# v2 predates; ours must be the default readout=none) -- the rule of v5_stage1.provenance
from core.tools.v5_stage1 import _flat

for seed in SEEDS:
    ref = _flat(yaml.safe_load((A2_RUNS / f's{seed}' / 'config.yaml').read_text()))
    for arm in ARMS:
        got = _flat(yaml.safe_load((ROOT / arm / f's{seed}' / 'config.yaml').read_text()))
        diff = {k: (ref.get(k), got.get(k)) for k in set(ref) | set(got)
                if not k.startswith(('stp.', 'v5.')) and ref.get(k) != got.get(k)}
        assert not diff, f'{arm} s{seed} differs from v2 A2 beyond stp: {diff}'
        assert got['v5.readout'] == constants.V5_READOUT_OFF, got['v5.readout']
        want = {f'stp.{k}': v for k, v in constants.STP_ARMS[arm].items()}
        assert all(got[k] == v for k, v in want.items()), (arm, {k: got[k] for k in want})
        print(f'{arm} s{seed}: config == v2 A2 s{seed} except', {k: v for k, v in got.items() if k.startswith('stp.')})

## 7. DoTA-CAP-dev — protocol B per arm (GPU)

`s1[::3]`, whole clip, per-clip min-max, native frames, macro over two-class clips. Input = v2's A2 bake of CAP-dev
(Nexar statistics) + the arm's patch cache. Skipped when the arm's read already covers `SEEDS`.

In [ ]:
PB_OUT = ROOT / 'eval' / 'dota' / CAP_DEV
D_INPUTS = {'dota_a2': (DOTA_A2, LOCAL / 'stage' / 'dota_a2', CAP_IDS),
            'dota_s1': (DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
            'dota_lab': (DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels', None),
            'patch_d': (PATCH_D, LOCAL / 'stage' / 'patch_dota', None)}


def d_staged(key):
    if key not in _STAGED:
        _STAGED[key] = stage(*D_INPUTS[key])
    return _STAGED[key]


def read_covers(path):
    if not path.exists():
        return False
    return sorted(json.loads(path.read_text()).get('per_run_macro', {})) == sorted(f's{s}' for s in SEEDS)


wrote = False
for arm in ARMS:
    out = PB_OUT / arm
    if read_covers(out / 'protocol_b_readout.json'):
        print(f'dota {arm}: already read on seeds {SEEDS}')
        continue
    local, t0 = LOCAL / 'dota' / arm, time.time()
    runs = [x for s in SEEDS for x in ('--run', f's{s}', ROOT / arm / f's{s}' / 'checkpoint_last.pt')]
    patch = ['--patch-dir', d_staged('patch_d')] if constants.STP_ARMS[arm]['sa2'] else []
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', *runs,
         '--input-dir', d_staged('dota_a2'), *patch, '--s1-dir', d_staged('dota_s1'),
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', d_staged('dota_lab'), '--split', CAP_DEV, '--out-dir', local, '--device', 'auto'],
        log=LOCAL / f'dota_{arm}.log')
    publish(local, out)
    wrote = True
    r = json.loads((local / 'protocol_b_readout.json').read_text())
    print(f"dota {arm}: macro {r['macro']['mean']:.4f} per seed {r['per_run_macro']} ({(time.time() - t0) / 60:.1f} min)")
if wrote:
    flush_drive()
r1 = json.loads((R1_PB / 'protocol_b_readout.json').read_text())
print(f"control R1 (v5 Stage 2 read): macro {r1['macro']['mean']:.4f} per seed {r1['per_run_macro']}")
if A2_PB.exists():
    print('printed: v2 Nexar A2 per seed', json.loads(A2_PB.read_text()).get('per_run_macro'))

## 8. Read-out → **paste `REPORTS/stp_readout.md` to Claude**

Rule: `STP_BUILD.md` §3. CPU only. With one seed every sentence reads **PRELIMINARY (n = 1 seed)**.

In [ ]:
r1_seeds = sorted(json.loads((R1_PB / 'protocol_b_readout.json').read_text())['per_run_macro'])
assert all(f's{s}' in r1_seeds for s in SEEDS), f'R1 read covers {r1_seeds}, this run needs {SEEDS}'
arms = [x for a in ARMS for x in ('--arm', a, PB_OUT / a)]
run([sys.executable, '-m', 'core.tools.stp_readout', '--control', CONTROL, R1_PB, *arms,
     '--seeds', *SEEDS, '--runs-root', ROOT, '--metadata', DOTA / 'metadata_val.json',
     '--split-file', DOTA / 'val_split.txt', '--out-dir', REPORTS])
print((REPORTS / 'stp_readout.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/stp_readout.md to Claude')